# Modul 06 · Notebook 02 — NVIDIA NIM (Generator di Cloud)

Model 70B seperti **Llama-3.3-70B** butuh puluhan GB memori — **tidak muat** di T4. Solusinya: jangan jalankan di GPU-mu, **panggil dari cloud**.

**NVIDIA NIM** (*NVIDIA Inference Microservices*) menyajikan model besar lewat API yang **kompatibel-OpenAI** di `https://integrate.api.nvidia.com/v1`. Artinya: kode yang sama yang kamu pakai untuk OpenAI bisa langsung memanggil model NVIDIA — **tanpa GPU di sisimu**.

> 🔑 **Sebelum mulai:** ambil API key gratis di [build.nvidia.com](https://build.nvidia.com), lalu tambahkan ke **Colab Secrets** (ikon 🔑 di kiri) dengan nama **`NVIDIA_API_KEY`**, dan aktifkan akses untuk notebook ini.

In [1]:
# Instal openai + ambil helper bersama (nvidia_utils.py) dari repo
!pip -q install openai
import os
if not os.path.exists("nvidia_utils.py"):
    !wget -q https://raw.githubusercontent.com/chmdznr/navasena-gen-ml-course/module06-rework/06_nvidia_ecosystem/tools/nvidia_utils.py

In [2]:
# Ambil key dari Colab Secrets (JANGAN pernah menaruh key langsung di kode)
import os
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
print("Key termuat:", bool(os.environ.get("NVIDIA_API_KEY")))

Key termuat: True


In [3]:
# Panggil model 70B di cloud lewat helper nim_client()
from nvidia_utils import nim_client
client = nim_client()                       # default -> https://integrate.api.nvidia.com/v1
MODEL = "meta/llama-3.3-70b-instruct"

r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user",
               "content": "Apa itu retrieval-augmented generation (RAG)? Jawab singkat dalam Bahasa Indonesia."}],
)
print(r.choices[0].message.content)

Retrieval-augmented generation (RAG) adalah teknik yang menggabungkan pencarian (retrieval) dan generasi (generation) untuk memperbaiki kualitas teks yang dihasilkan oleh model bahasa. Teknik ini menggunakan pencarian untuk menemukan informasi relevan dan kemudian menggunakannya untuk menghasilkan teks yang lebih akurat dan informatif.


## Kenapa "kompatibel-OpenAI" itu penting?

Karena kode aplikasimu jadi **portabel**. Target inference (Ollama lokal, vLLM, NVIDIA NIM) semua bicara protokol yang sama — pindah target cukup ganti `base_url`. Ini persis pola yang kamu lihat di Modul 04 (deployment SLM): satu kode klien, banyak backend.

In [4]:
# Satu key, dua peran: model yang sama bisa jadi "juri" (LLM-as-judge)
jawaban = "RAG adalah teknik yang menggabungkan pencarian dokumen dengan LLM untuk menjawab berdasarkan sumber."
penilaian = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user",
               "content": f"Beri skor 1-5 untuk kejelasan jawaban ini (balas ANGKA saja): '{jawaban}'"}],
)
print("Skor juri:", penilaian.choices[0].message.content.strip())

Skor juri: 5


## 🧪 Latihan

1. Ganti `MODEL` ke model NIM lain dari katalog [build.nvidia.com](https://build.nvidia.com) (mis. model Mistral atau Qwen) — bandingkan gaya jawabannya.
2. Tambah parameter `temperature=0.2` pada `create(...)` — apakah jawaban jadi lebih konsisten?
3. Pikirkan: kenapa LLM-as-judge berguna untuk **mengukur** kualitas (ingat Modul 04 evaluation)? Konsep ini kembali di **nb04** sebagai *self-check rail*.